# 10｜手动控制与标准速度


> 运行方式：从上到下执行代码单元。控制类 Notebook **没有布尔安全开关**；执行控制单元就会把任务提交给 SDK。
> 需要仿真的课程请先在 FleetDock IDEA 中启动仿真，并确认 `drone1`、`drone2` 等设备在线。


本课会让 `drone1` 起飞，向前飞行 2 秒，释放输入后返航。

![协同与智能能力](assets/advanced.svg)


## ManualControlSession

键盘、摇杆和遥控面板只需不断提交标准机体系速度。SDK 负责刷新频率、限速、输入超时和释放动作。


In [ ]:
import time
from fleetdock_runtime import FleetDock
from fleetdock import ManualControlOptions

FleetDock.init()
drone = FleetDock.devices.drone1
print(drone.connect())
print(
    FleetDock.core.plan(drone)
    .wait_until_ready(timeout_s=60.0)
    .unlock()
    .takeoff(altitude=10.0)
    .run(step_timeout_s=120.0)
)


In [ ]:
options = ManualControlOptions(
    max_horizontal_speed_m_s=3.0,
    max_vertical_speed_m_s=1.0,
    max_yaw_rate_deg_s=45.0,
)

with drone.manual_control(options) as control:
    control.update(forward_m_s=2.0)
    time.sleep(2.0)
    control.release()
    print(control.status())


In [ ]:
print(
    FleetDock.core.plan(drone)
    .return_to_launch()
    .run(step_timeout_s=120.0)
)
FleetDock.close()


## 重点

`update()` 接收“前、右、上、偏航角速度”四个标准量。应用无需自己编写 Offboard 刷新线程或输入超时逻辑。
